<a href="https://www.kaggle.com/code/eduardopalves/pld-aml-detector-detec-o-de-lavagem-de-dinheiro?scriptVersionId=321362703" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# PLD-AML Detector

Detecção de lavagem de dinheiro com modelos **não supervisionados** aplicados ao dataset sintético PaySim.

> **Objetivo:** Validar o uso de modelos não supervisionados em cenários onde não há labels históricos confiáveis — e documentar de forma transparente onde essa abordagem funciona bem e onde tem limitações.

---

## Sumário

- [Contexto e motivação](#contexto-e-motivação)
- [Arquitetura do pipeline](#arquitetura-do-pipeline)
- [Resultados obtidos](#resultados-obtidos)
- [Limitações conhecidas](#limitações-conhecidas)
- [Estrutura do projeto](#estrutura-do-projeto)
- [Como executar](#como-executar)
- [API REST](#api-rest)
- [Testes](#testes)
- [Dataset](#dataset)

---

## Contexto e motivação

Em operações de PLD (Prevenção à Lavagem de Dinheiro), é comum não ter acesso a labels históricos confiáveis — seja por ausência de registros de investigações anteriores, seja por restrições regulatórias. Nesse cenário, modelos supervisionados não são viáveis.

Este projeto explora até onde modelos **não supervisionados** conseguem chegar:

- **Isolation Forest** — detecta anomalias por isolamento de pontos no espaço de features
- **Autoencoder** — aprende a reconstruir transações normais; fraudes geram erro de reconstrução alto
- **Ensemble** — combina os dois modelos com thresholds adaptativos por tipo de transação
- **SHAP** — explica quais features influenciam cada decisão (requisito regulatório)
- **Análise de Grafo** — identifica padrões de rede como smurfing, layering e hubs suspeitos

---

## Arquitetura do pipeline

```
CSV PaySim
    │
    ▼
┌─────────────────────────────────────────────────────┐
│  Feature Engineering  (28 features)                 │
│  Base · Tipo · Conta · Balanço · Velocidade · Tempo │
└────────────────────┬────────────────────────────────┘
                     │
          ┌──────────┴──────────┐
          ▼                     ▼
┌──────────────────┐   ┌─────────────────────────────┐
│ Isolation Forest │   │ Autoencoder  (Weighted MSE) │
│  n_estimators=300│   │ 28→64→32→16→8→16→32→64→28  │
│  contamination=y │   │ BatchNorm · Dropout(0.15)   │
└────────┬─────────┘   └──────────────┬──────────────┘
         │                            │
         │             ┌──────────────▼──────────────┐
         │             │  Threshold Adaptativo        │
         │             │  TRANSFER: ~0.26             │
         │             │  CASH_OUT: ~0.31             │
         └─────────────┴──────────────┬──────────────┘
                                      │
                             ┌────────▼────────┐
                             │    Ensemble      │
                             │  AE + IF low-val │
                             └────────┬────────┘
                                      │
                    ┌─────────────────┼──────────────────┐
                    ▼                 ▼                   ▼
              SHAP Values      Graph Analysis       API REST
              (explicação)     (rede suspeita)    (FastAPI)
```

### Features engineered (28)

| Grupo | Features |
|---|---|
| Base | `log_amount`, `type_enc` |
| Balanços | `diff_orig`, `diff_dest`, `balance_error`, `orig_zeroed`, `full_drain` |
| Ratios | `balance_error_ratio`, `amount_ratio_dest`, `balance_retention_orig` |
| Conta | `dest_is_customer`, `orig_to_customer`, `dest_had_zero`, `dest_zeroed_after` |
| Tipo | `is_fraud_type`, `is_transfer`, `is_cashout` |
| Velocidade | `velocity_orig`, `velocity_dest`, `log_volume_orig` |
| Temporal | `hour_of_day`, `day_of_sim`, `off_hours`, `step` |
| Raw | `oldbalanceOrg`, `newbalanceOrig`, `oldbalanceDest`, `newbalanceDest` |

---

## Resultados obtidos

Avaliação no dataset PaySim (200k transações, 4.1% fraude).

### Métricas principais

| Métrica | Valor | Meta | Status |
|---|---|---|---|
| AUC-ROC Autoencoder | 0.9609 | ≥ 0.95 | ✓ |
| AUC-ROC Isolation Forest | 0.9091 | — | ✓ |
| F1 fraude | 0.578 | ≥ 0.55 | ✓ |
| Precision fraude | 0.481 | ≥ 0.50 | ✗ |
| Recall fraude | 0.725 | — | — |
| Razão erro fraude/normal | 67.6x | ≥ 50x | ✓ |

### Detecção por tipo de transação

| Tipo | Detectadas | Perdidas | Taxa | Threshold |
|---|---|---|---|---|
| TRANSFER | 3.620 | 477 | **88.4%** | ~0.257 |
| CASH_OUT | 2.335 | 1.781 | **56.7%** | ~0.306 |

### Threshold adaptativo

Ao usar thresholds separados por tipo de transação:

| Tipo | F1 | Precision | Recall |
|---|---|---|---|
| TRANSFER | 0.766 | 0.631 | **0.974** |
| CASH_OUT | 0.586 | 0.609 | 0.565 |
| **Global adaptativo** | **0.593** | — | — |

### Valor financeiro

| | Total | Valor médio |
|---|---|---|
| Fraudes detectadas (TP) | R$ 11,4 bilhões | R$ 1,9 milhão |
| Fraudes perdidas (FN) | R$ 625 milhões | R$ 277 mil |

### SHAP — features mais importantes

**Isolation Forest:** `off_hours` > `orig_zeroed` > `is_transfer` > `full_drain`

**Autoencoder:** `diff_orig` > `newbalanceDest` > `balance_error` > `oldbalanceDest`

---

## Limitações conhecidas

Esta seção documenta de forma transparente os pontos fracos identificados — parte essencial da validação da abordagem não supervisionada.

### 1. CASH_OUT de baixo valor — principal ponto cego

Fraudes CASH_OUT com valor abaixo de ~R$200k têm taxa de detecção de apenas **56.7%**. A análise de features mostra que essas transações têm `full_drain=0.9996` e `orig_zeroed=0.9935` — praticamente idênticos às fraudes detectadas — o que indica que o padrão é genuinamente difícil de separar sem labels.

### 2. Variabilidade entre treinos

Por ser não supervisionado, o autoencoder pode convergir para soluções diferentes a cada execução. Foi implementado multi-run (3 seeds) com seleção pelo menor val_loss, mas a variabilidade persiste:

| Execução | F1 | Razão erro | Precision |
|---|---|---|---|
| Run 1 | 0.643 | 34.4x | 0.569 |
| Run 2 | 0.582 | 18.9x | 0.640 |
| Run 3 | 0.578 | 67.6x | 0.481 |

### 3. Precision abaixo da meta

Com threshold otimizado para F1, a precision fica em torno de 0.48–0.57. Com threshold adaptativo por tipo, melhora para 0.609–0.631. Em produção, é necessário calibrar o trade-off entre recall (não deixar fraudes passarem) e precision (custo operacional de investigação).

### 4. Grafo limitado pela amostra

A análise de grafo usa uma amostra de 200k de 6,3M transações. Padrões de smurfing e layering que envolvem contas em diferentes partes do dataset podem não aparecer na amostra. Os 113 hubs de alta centralidade identificados são o resultado mais robusto.

### 5. Quando usar e quando não usar

| Cenário | Recomendação |
|---|---|
| Sem labels históricos | ✓ Use esta abordagem |
| Labels confiáveis disponíveis | Prefira modelo supervisionado (XGBoost, RF) |
| Fraudes novas / nunca vistas | ✓ Não supervisionado generaliza melhor |
| Alta precisão regulatória necessária | Combine com regras de negócio |

---

## Estrutura do projeto

```
pld-aml-detector/
├── src/aml_detector/
│   ├── config.py                  # Hiperparâmetros e constantes
│   ├── data/
│   │   └── loader.py              # Carga e limpeza do PaySim
│   ├── features/
│   │   └── engineering.py         # 28 features engineered
│   ├── models/
│   │   ├── autoencoder.py         # Autoencoder + weighted MSE + multi-run
│   │   ├── isolation_forest.py    # Isolation Forest
│   │   ├── ensemble.py            # Threshold adaptativo por tipo
│   │   └── persistence.py         # Salvar/carregar artefatos
│   ├── evaluation/
│   │   └── metrics.py             # ROC, PR, F1, análise de erros
│   ├── explainability/
│   │   └── shap_explain.py        # SHAP para IF e Autoencoder
│   ├── graph/
│   │   └── analysis.py            # NetworkX + métricas + PyVis
│   └── api/
│       ├── app.py                 # FastAPI + lifespan
│       ├── routes.py              # /predict, /batch, /health
│       ├── schemas.py             # Pydantic schemas
│       └── model_store.py         # Singleton de modelos em memória
├── scripts/
│   └── train.py                   # Pipeline completo CLI
├── notebooks/
│   └── evaluation.ipynb           # Avaliação completa (roda no Kaggle)
├── tests/
│   ├── conftest.py                # Fixtures sintéticas + --data flag
│   ├── test_features.py           # 21 testes de feature engineering
│   ├── test_models.py             # 19 testes de modelos
│   ├── test_contracts.py          # 22 testes de contratos de dados
│   ├── test_pipeline.py           # 8 testes end-to-end
│   └── test_real_data.py          # 22 testes com dados reais (@real_data)
└── pyproject.toml
```

---

## Como executar

### No Kaggle (recomendado — sem instalação local)

1. Acesse [kaggle.com](https://www.kaggle.com) e crie um novo notebook
2. Adicione o dataset `ealaxi/paysim1` pelo painel direito
3. Menu **File → Import Notebook → GitHub** → cole a URL deste repositório
4. Selecione `notebooks/evaluation.ipynb` e execute

# AML Detector — Notebook de Avaliação

Execute célula a célula para obter um relatório completo sobre:

| Seção | Pergunta respondida |
|---|---|
| 1. Configuração | Onde estão os dados e artefatos? |
| 2. Validação dos dados | O dataset é compatível com o pipeline? |
| 3. Auditoria do pipeline | Quais tratativas foram aplicadas? |
| 4. Treinamento / Carga | Treinar do zero ou usar modelo salvo? |
| 5. Desempenho | AUC-ROC, F1, Precision, Recall |
| 6. Análise de erros | Por que o modelo perde certas fraudes? |
| 7. SHAP | Quais features mais influenciam cada decisão? |
| 8. Grafo | Padrões de rede suspeitos |
| 9. Resumo | Conclusões e recomendações |

## 1 · Configuração

In [ ]:
import os, sys

# ── Detecção de ambiente ──────────────────────────────────────────────────
ON_KAGGLE = os.path.exists("/kaggle/input")

if ON_KAGGLE:
    # Clona o repositório e instala o pacote (executa só uma vez)
    if not os.path.exists("/kaggle/working/pld-aml-detector"):
        os.system("git clone -q https://github.com/Eduselva/pld-aml-detector.git /kaggle/working/pld-aml-detector")
        os.system("cd /kaggle/working/pld-aml-detector && git checkout -q claude/continue-work-wax36")
        os.system("pip install -q -e /kaggle/working/pld-aml-detector/.[dev,api]")
    sys.path.insert(0, "/kaggle/working/pld-aml-detector/src")
    # Caminhos Kaggle
    # Descoberta automática do CSV PaySim
    import glob
    _candidates = glob.glob("/kaggle/input/**/PS_*.csv", recursive=True) + \
                  glob.glob("/kaggle/input/**/*paysim*.csv", recursive=True) + \
                  glob.glob("/kaggle/input/**/*PS_*.csv", recursive=True)
    _candidates = list(dict.fromkeys(_candidates))  # deduplica mantendo ordem
    if not _candidates:
        raise FileNotFoundError(
            "Dataset PaySim não encontrado em /kaggle/input/. "
            "Adicione o dataset 'ealaxi/paysim1' pelo painel direito do Kaggle."
        )
    DATA_PATH = _candidates[0]
    print(f"CSV encontrado: {DATA_PATH}")
    ARTIFACTS_DIR  = "/kaggle/working/models"
else:
    sys.path.insert(0, "../src")
    # Caminhos locais
    DATA_PATH      = "../data/PS_20174392719_1491204439457_log.csv"
    ARTIFACTS_DIR  = "../models"

print(f"Ambiente : {'Kaggle' if ON_KAGGLE else 'Local'}")
print(f"Dataset  : {DATA_PATH}")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Ajuste estas opções conforme necessário ───────────────────────────────
LOAD_MODEL   = False   # True = carrega artefatos salvos; False = treina do zero
RUN_SHAP     = True    # SHAP pode demorar ~1 min
RUN_GRAPH    = True    # Análise de grafo
# ─────────────────────────────────────────────────────────────────────────

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

from aml_detector.config import FEATURE_COLS, RANDOM_SEED
import tensorflow as tf
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
print("✓ Ambiente pronto")


## 2 · Validação dos dados de entrada

In [ ]:
from aml_detector.data.loader import load_paysim

df = load_paysim(DATA_PATH)

REQUIRED_COLS = {
    "step", "type", "amount",
    "nameOrig", "oldbalanceOrg", "newbalanceOrig",
    "nameDest",  "oldbalanceDest", "newbalanceDest",
    "isFraud",
}
missing = REQUIRED_COLS - set(df.columns)
assert not missing, f"Colunas faltando: {missing}"

fraud_rate = df["isFraud"].mean()
print(f"\n{'─'*55}")
print(f"  VALIDAÇÃO DO DATASET")
print(f"{'─'*55}")
print(f"  Linhas          : {len(df):>10,}")
print(f"  Fraudes         : {df['isFraud'].sum():>10,}  ({fraud_rate*100:.3f}%)")
print(f"  Tipos presentes : {sorted(df['type'].unique())}")
print(f"  Steps           : {df['step'].min()} → {df['step'].max()}")
print(f"  Colunas OK      : {sorted(REQUIRED_COLS)}")
assert 0.001 <= fraud_rate <= 0.15, f"Taxa de fraude fora do esperado: {fraud_rate:.4f}"
fraud_types = set(df[df["isFraud"]==1]["type"].unique())
assert fraud_types.issubset({"TRANSFER", "CASH_OUT"}), f"Tipos inesperados: {fraud_types}"
print(f"  Tipos com fraude: {fraud_types}  ✓")
print(f"{'─'*55}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Distribuições do Dataset", fontsize=13)

# Distribuição log(amount) por label
for label, color, name in [(0, "steelblue", "Normal"), (1, "tomato", "Fraude")]:
    df[df["isFraud"]==label]["amount"].apply(np.log1p).hist(
        bins=60, ax=axes[0], alpha=0.7, color=color, label=name, density=True
    )
axes[0].set_title("log(amount) por label")
axes[0].legend()

# Volume por tipo
df.groupby("type")["amount"].sum().sort_values().plot(
    kind="barh", ax=axes[1], color="steelblue"
)
axes[1].set_title("Volume total por tipo")

# Taxa de fraude por tipo
(df.groupby("type")["isFraud"].mean() * 100).sort_values().plot(
    kind="barh", ax=axes[2], color="tomato"
)
axes[2].set_title("Taxa de fraude por tipo (%)")

plt.tight_layout()
plt.show()

## 3 · Auditoria do Pipeline de Features

In [ ]:
from aml_detector.features.engineering import build_features, scale_features

X, y = build_features(df)
X_scaled, scaler = scale_features(X)

assert X.isnull().sum().sum() == 0, "Nulos encontrados após feature engineering"
assert np.isfinite(X.values).all(), "Infinitos encontrados"

print(f"{'─'*65}")
print(f"  AUDITORIA DE FEATURES")
print(f"{'─'*65}")
print(f"  Shape           : {X.shape}")
print(f"  Fraudes         : {y.sum():,} ({y.mean()*100:.3f}%)")
print(f"  Nulos           : {X.isnull().sum().sum()}  ✓")
print(f"  Infinitos       : {(~np.isfinite(X.values)).sum()}  ✓")
print(f"\n  {'Feature':<28} {'Média':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print(f"  {'─'*68}")
for col in FEATURE_COLS:
    s = X[col]
    print(f"  {col:<28} {s.mean():>10.3f} {s.std():>10.3f} {s.min():>10.3f} {s.max():>10.3f}")
print(f"{'─'*65}")

In [ ]:
# Correlação de cada feature com isFraud
corr = X.assign(isFraud=y).corr()["isFraud"].drop("isFraud").sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["tomato" if v > 0 else "steelblue" for v in corr]
ax.barh(corr.index, corr.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Correlação de Pearson de cada feature com isFraud")
ax.set_xlabel("Correlação")
plt.tight_layout()
plt.show()

## 4 · Treinamento / Carga do Modelo

In [ ]:
# Override do diretório de artefatos (Kaggle vs local)
import aml_detector.config as _cfg, pathlib
_cfg.MODELS_DIR = pathlib.Path(ARTIFACTS_DIR)
_cfg.MODELS_DIR.mkdir(parents=True, exist_ok=True)

from aml_detector.models.isolation_forest import train_isolation_forest, iso_scores
from aml_detector.models.autoencoder import train_autoencoder, reconstruction_errors
from aml_detector.models.ensemble import find_optimal_threshold, train_low_value_iso, ensemble_predict
from aml_detector.models.persistence import load_artifacts, save_artifacts, artifacts_exist

if LOAD_MODEL and artifacts_exist():
    print("Carregando artefatos salvos...")
    scaler, ae, metadata = load_artifacts()
    best_thr = metadata["threshold"]
    # Recalcula scores com o scaler salvo
    X_scaled = scaler.transform(X)
    scores_ae  = reconstruction_errors(ae, X_scaled)
    iso        = train_isolation_forest(X_scaled, contamination=float(y.mean()))
    scores_iso = iso_scores(iso, X_scaled)
    print(f"✓ Modelo carregado — threshold={best_thr:.6f}")
else:
    print("Treinando do zero...")
    iso = train_isolation_forest(X_scaled, contamination=float(y.mean()))
    scores_iso = iso_scores(iso, X_scaled)

    ae, history = train_autoencoder(X_scaled, y)
    scores_ae   = reconstruction_errors(ae, X_scaled)

    best_thr, best_f1 = find_optimal_threshold(scores_ae, y)
    print(f"\n✓ Treino concluído — threshold ótimo={best_thr:.6f}  F1={best_f1:.3f}")

In [ ]:
# Curva de treino (só quando treina do zero)
if not LOAD_MODEL:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history.history["loss"],     label="Treino",    color="steelblue")
    ax.plot(history.history["val_loss"], label="Validação", color="coral")
    ax.set_title("Autoencoder — Curva de Loss (Weighted MSE)")
    ax.set_xlabel("Época")
    ax.set_ylabel("Loss")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5 · Avaliação de Desempenho

In [ ]:
from sklearn.metrics import (
    roc_auc_score, classification_report,
    RocCurveDisplay, precision_recall_curve, auc
)

auc_iso = roc_auc_score(y, scores_iso)
auc_ae  = roc_auc_score(y, scores_ae)

error_ratio = scores_ae[y==1].mean() / (scores_ae[y==0].mean() + 1e-9)

print(f"{'─'*50}")
print(f"  RESUMO DE DESEMPENHO")
print(f"{'─'*50}")
print(f"  AUC-ROC  Isolation Forest : {auc_iso:.4f}")
print(f"  AUC-ROC  Autoencoder      : {auc_ae:.4f}")
print(f"  Erro médio — normais      : {scores_ae[y==0].mean():.4f}")
print(f"  Erro médio — fraudes      : {scores_ae[y==1].mean():.4f}")
print(f"  Razão fraude/normal       : {error_ratio:.1f}x")
print(f"  Threshold ótimo (MSE)     : {best_thr:.6f}")
print(f"{'─'*50}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC
RocCurveDisplay.from_predictions(y, scores_iso, name=f"Isolation Forest (AUC={auc_iso:.3f})",
                                  ax=axes[0], color="steelblue")
RocCurveDisplay.from_predictions(y, scores_ae,  name=f"Autoencoder      (AUC={auc_ae:.3f})",
                                  ax=axes[0], color="tomato")
axes[0].plot([0,1],[0,1],"k--",lw=0.8)
axes[0].set_title("ROC — Comparação")

# Distribuição de scores — clipa em percentil 99 para evitar escala dominada por outliers
for ax, scores, title in [
    (axes[1], scores_iso, "Isolation Forest"),
    (axes[2], scores_ae,  "Autoencoder (MSE)"),
]:
    cap = np.percentile(scores, 99)
    scores_clipped = np.clip(scores, None, cap)
    ax.hist(scores_clipped[y==0], bins=80, alpha=0.6, label="Normal",  color="steelblue", density=True)
    ax.hist(scores_clipped[y==1], bins=80, alpha=0.7, label="Fraude",  color="tomato",    density=True)
    if title == "Autoencoder (MSE)":
        ax.axvline(best_thr, color="black", linestyle="--", lw=1.2, label=f"Threshold={best_thr:.3f}")
    ax.set_title(f"Scores — {title}")
    ax.set_xlabel(f"Score (cap p99={cap:.3f})")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# Curva Precision-Recall + F1 por threshold
precision, recall, thresholds = precision_recall_curve(y, scores_ae)
pr_auc = auc(recall, precision)
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9)
best_idx = f1_scores.argmax()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(recall, precision, color="tomato", lw=1.5, label=f"AUC-PR = {pr_auc:.3f}")
axes[0].scatter(recall[best_idx], precision[best_idx], color="black", zorder=5,
                label=f"F1 ótimo = {f1_scores[best_idx]:.3f}")
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision")
axes[0].set_title("Autoencoder — Curva Precision-Recall")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(thresholds, f1_scores, color="steelblue", lw=1.5)
axes[1].axvline(best_thr, color="tomato", linestyle="--", lw=1,
                label=f"Threshold ótimo = {best_thr:.4f}\nF1 = {f1_scores[best_idx]:.3f}")
axes[1].set_xlabel("Threshold (MSE)"); axes[1].set_ylabel("F1-score")
axes[1].set_title("F1 por Threshold — Autoencoder")
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_xlim(0, best_thr * 5)

plt.tight_layout()
plt.show()

# Classification reports
y_pred_ae  = (scores_ae  > best_thr).astype(int)
y_pred_iso = (scores_iso > np.percentile(scores_iso, (1-y.mean())*100)).astype(int)

print("── Autoencoder (threshold ótimo) ──")
print(classification_report(y, y_pred_ae, target_names=["Normal","Fraude"]))
print("── Isolation Forest ──")
print(classification_report(y, y_pred_iso, target_names=["Normal","Fraude"]))

In [ ]:
# ── Threshold adaptativo por tipo ─────────────────────────────────────────
from aml_detector.models.ensemble import find_thresholds_by_type, predict_with_type_thresholds
from sklearn.metrics import f1_score, precision_score, recall_score

types_series = df["type"].reset_index(drop=True)
thresholds_by_type, global_thr = find_thresholds_by_type(scores_ae, y.values, types_series)

y_pred_adaptive = predict_with_type_thresholds(scores_ae, types_series, thresholds_by_type, global_thr)

print(f"{'─'*58}")
print(f"  THRESHOLD ADAPTATIVO POR TIPO")
print(f"{'─'*58}")
print(f"  {'Tipo':<12} {'Threshold':>12} {'F1':>8} {'Precision':>10} {'Recall':>8}")
print(f"  {'─'*56}")
for tx_type, thr in sorted(thresholds_by_type.items()):
    mask = types_series == tx_type
    y_t = y[mask].values
    s_t = scores_ae[mask]
    pred_t = (s_t > thr).astype(int)
    f1_t  = f1_score(y_t, pred_t, zero_division=0)
    pr_t  = precision_score(y_t, pred_t, zero_division=0)
    re_t  = recall_score(y_t, pred_t, zero_division=0)
    print(f"  {tx_type:<12} {thr:>12.6f} {f1_t:>8.3f} {pr_t:>10.3f} {re_t:>8.3f}")

print(f"  {'─'*56}")
f1_global   = f1_score(y, y_pred_ae,       zero_division=0)
f1_adaptive = f1_score(y, y_pred_adaptive,  zero_division=0)
print(f"  Global threshold   : {best_thr:.6f}  →  F1 global  = {f1_global:.3f}")
print(f"  Adaptive threshold :  por tipo   →  F1 adaptativo = {f1_adaptive:.3f}")
delta = (f1_adaptive - f1_global) * 100
print(f"  Ganho F1           : {delta:+.1f} p.p.")
print(f"{'─'*58}")


## 6 · Análise de Erros — Por que o modelo perde certas fraudes?

In [ ]:
results = pd.DataFrame({
    "ae_score": scores_ae,
    "pred":     y_pred_ae,
    "true":     y.values,
}, index=X.index).join(X)
results["type"] = df["type"]

tp = results[(results["true"]==1) & (results["pred"]==1)]
fn = results[(results["true"]==1) & (results["pred"]==0)]
fp = results[(results["true"]==0) & (results["pred"]==1)]

print(f"{'─'*55}")
print(f"  ANÁLISE DE ERROS")
print(f"{'─'*55}")
print(f"  Verdadeiros Positivos (TP) : {len(tp):>6,}  ({len(tp)/y.sum()*100:.1f}% das fraudes)")
print(f"  Falsos Negativos     (FN) : {len(fn):>6,}  ({len(fn)/y.sum()*100:.1f}% das fraudes)")
print(f"  Falsos Positivos     (FP) : {len(fp):>6,}")

print(f"\n  Tipo de transação: detectadas vs perdidas")
comp = pd.DataFrame({
    "Detectadas (TP)": tp["type"].value_counts(),
    "Perdidas (FN)": fn["type"].value_counts(),
}).fillna(0).astype(int)
comp["Taxa detecção"] = (comp["Detectadas (TP)"] /
                         (comp["Detectadas (TP)"] + comp["Perdidas (FN)"]) * 100).round(1)
print(comp)
print(f"{'─'*55}")

In [ ]:
# Features médias: TP vs FN
feat_cols = ["log_amount", "diff_orig", "balance_error", "balance_error_ratio",
             "full_drain", "orig_zeroed", "dest_had_zero", "velocity_orig"]

comparison = pd.DataFrame({
    "TP (detectadas)": tp[feat_cols].mean(),
    "FN (perdidas)":   fn[feat_cols].mean(),
})
comparison["Δ (TP-FN)"] = comparison["TP (detectadas)"] - comparison["FN (perdidas)"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribuição do score AE: TP vs FN
axes[0].hist(tp["ae_score"], bins=50, alpha=0.7, color="seagreen", density=True, label="TP")
axes[0].hist(fn["ae_score"], bins=50, alpha=0.7, color="tomato",   density=True, label="FN")
axes[0].axvline(best_thr, color="black", linestyle="--", lw=1, label="Threshold")
axes[0].set_title("Score AE: fraudes detectadas vs perdidas")
axes[0].set_xlabel("MSE"); axes[0].legend()

# Delta de features
delta = comparison["Δ (TP-FN)"]
colors = ["seagreen" if v > 0 else "tomato" for v in delta]
axes[1].barh(delta.index, delta.values, color=colors)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Δ features médias  (TP − FN)\n+ = detectadas têm valor maior")

plt.tight_layout()
plt.show()

print("\nFeatures médias por grupo:")
print(comparison.round(4))

In [ ]:
# Distribuição de log(amount): TP, FN, FP
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, groups, title in [
    (axes[0],
     [(tp, "TP (fraudes detectadas)", "seagreen"),
      (fn, "FN (fraudes perdidas)",   "tomato")],
     "Distribuição log(amount): TP vs FN"),
    (axes[1],
     [(fp, "FP (falsos alarmes)",     "orange"),
      (results[results["true"]==0], "Normais", "steelblue")],
     "Distribuição log(amount): FP vs Normais"),
]:
    for grp, label, color in groups:
        grp["log_amount"].hist(bins=50, ax=ax, alpha=0.65, color=color,
                               density=True, label=f"{label} (n={len(grp):,})")
    ax.set_title(title); ax.set_xlabel("log(amount)"); ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 7 · SHAP — Explicabilidade

In [ ]:
if RUN_SHAP:
    import shap
    from aml_detector.explainability.shap_explain import (
        explain_isolation_forest, plot_iso_shap_summary, plot_iso_shap_beeswarm,
        explain_autoencoder, plot_ae_shap_summary, plot_ae_shap_delta,
    )

    print("── SHAP Isolation Forest ──")
    iso_explainer, iso_sv = explain_isolation_forest(iso, X_scaled)
    plot_iso_shap_summary(iso_sv, X_scaled, save=False)
    plt.show()
    plot_iso_shap_beeswarm(iso_sv, X_scaled, save=False)
    plt.show()
else:
    print("RUN_SHAP=False — pulando. Mude para True para ver os gráficos.")

In [ ]:
if RUN_SHAP:
    print("── SHAP Autoencoder ──")
    ae_explainer, ae_sv, eval_idx, top_idx = explain_autoencoder(ae, X_scaled, scores_ae, y)
    plot_ae_shap_summary(ae_sv, X_scaled, eval_idx, save=False)
    plt.show()
    plot_ae_shap_delta(ae_sv, eval_idx, top_idx, save=False)
    plt.show()

## 8 · Análise de Grafo

In [ ]:
if RUN_GRAPH:
    from aml_detector.graph.analysis import (
        build_graph, compute_graph_metrics, plot_ego_networks, export_pyvis
    )

    G = build_graph(df, scores_ae)
    df_metrics = compute_graph_metrics(G)

    # Distribuições de métricas: fraud vs normal
    metrics_cols = [
        ("fan_out_score",    "Fan-out (smurfing)"),
        ("fan_in_score",     "Fan-in (aggregation)"),
        ("pagerank",         "PageRank"),
        ("sent_vs_received", "Desequilíbrio enviado/recebido"),
        ("graph_risk_score", "Graph Risk Score"),
    ]
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    fig.suptitle("Métricas de Grafo — Fraud vs Normal", fontsize=13)
    for ax, (col, label) in zip(axes, metrics_cols):
        cap = df_metrics[col].quantile(0.99)
        ax.hist(df_metrics[df_metrics["is_fraud"]==0][col].clip(0, cap),
                bins=50, alpha=0.6, color="steelblue", density=True, label="Normal")
        ax.hist(df_metrics[df_metrics["is_fraud"]==1][col].clip(0, cap),
                bins=50, alpha=0.7, color="tomato",    density=True, label="Fraud")
        ax.set_title(label, fontsize=8); ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()

    plot_ego_networks(G, df_metrics, save=False)
    plt.show()

    export_pyvis(G, df_metrics)
else:
    print("RUN_GRAPH=False — pulando.")

## 9 · Resumo e Recomendações

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

prec = precision_score(y, y_pred_ae, zero_division=0)
rec  = recall_score(y, y_pred_ae, zero_division=0)
f1   = f1_score(y, y_pred_ae, zero_division=0)

STATUS_AUC  = "✓" if auc_ae  >= 0.95 else "✗"
STATUS_F1   = "✓" if f1      >= 0.55 else "✗"
STATUS_PREC = "✓" if prec    >= 0.50 else "✗"
STATUS_RATIO= "✓" if error_ratio >= 50 else "✗"

print("╔══════════════════════════════════════════════════════════╗")
print("║              RELATÓRIO FINAL DE AVALIAÇÃO               ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  AUC-ROC  Autoencoder      : {auc_ae:.4f}        {STATUS_AUC} (≥0.95)  ║")
print(f"║  AUC-ROC  Isolation Forest : {auc_iso:.4f}        {STATUS_AUC}          ║")
print(f"║  F1 fraude (threshold ótm) : {f1:.4f}        {STATUS_F1} (≥0.55)  ║")
print(f"║  Precision fraude          : {prec:.4f}        {STATUS_PREC} (≥0.50)  ║")
print(f"║  Recall fraude             : {rec:.4f}                    ║")
print(f"║  Razão erro fraude/normal  : {error_ratio:6.1f}x       {STATUS_RATIO} (≥50x)   ║")
print(f"║  Threshold ótimo (MSE)     : {best_thr:.6f}                ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Fraudes detectadas (TP)   : {len(tp):>6,}  ({len(tp)/y.sum()*100:.1f}%)           ║")
print(f"║  Fraudes perdidas   (FN)   : {len(fn):>6,}  ({len(fn)/y.sum()*100:.1f}%)           ║")
print(f"║  Falsos positivos   (FP)   : {len(fp):>6,}                    ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  DIAGNÓSTICO DOS FN:                                     ║")
print(f"║  · Valor médio FN : R${np.exp(fn['log_amount'].mean()):>12,.0f}              ║")
print(f"║  · Valor médio TP : R${np.exp(tp['log_amount'].mean()):>12,.0f}              ║")
print("║  · FNs têm amount menor → erro de reconstrução baixo    ║")
print("║  · Considerar threshold adaptativo por faixa de valor   ║")
print("╚══════════════════════════════════════════════════════════╝")

# Breakdown por tipo de transação
print()
print("╔══════════════════════════════════════════════════════════╗")
print("║              BREAKDOWN POR TIPO DE TRANSAÇÃO            ║")
print("╠══════════════════════════════════════════════════════════╣")
for tx_type in ["TRANSFER", "CASH_OUT"]:
    mask_type = results["type"] == tx_type if "type" in results.columns else (df["type"] == tx_type).values[:len(results)]
    tp_type = results[mask_type & (results["true"] == 1) & (results["pred"] == 1)]
    fn_type = results[mask_type & (results["true"] == 1) & (results["pred"] == 0)]
    total_fraud_type = len(tp_type) + len(fn_type)
    taxa = len(tp_type) / total_fraud_type * 100 if total_fraud_type > 0 else 0.0
    print(f"║  {tx_type:<8} → Detectadas: {len(tp_type):>4}  |  Perdidas: {len(fn_type):>4}  |  Taxa: {taxa:5.1f}% ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  VALOR EM RISCO NÃO DETECTADO                           ║")
fn_amount = np.expm1(fn["log_amount"])
tp_amount = np.expm1(tp["log_amount"])
print(f"║  · FN total : R$ {fn_amount.sum():>12,.0f}  (valor médio: R$ {fn_amount.mean():>8,.0f})    ║")
print(f"║  · TP total : R$ {tp_amount.sum():>12,.0f}  (valor médio: R$ {tp_amount.mean():>8,.0f})    ║")
print("╚══════════════════════════════════════════════════════════╝")

In [ ]:
# ── Exporta dados pré-computados para o dashboard Streamlit ───────────────
import json as _json, os as _os
import numpy as _np
from sklearn.metrics import roc_curve as _roc_curve, f1_score, precision_score, recall_score

_export_dir = "/kaggle/working/dashboard_data" if ON_KAGGLE else "../dashboard/data"
_os.makedirs(_export_dir, exist_ok=True)

# 1. Métricas principais
_json.dump({
    "auc_autoencoder": float(auc_ae), "auc_isolation_forest": float(auc_iso),
    "f1": float(f1), "precision": float(prec), "recall": float(rec),
    "error_ratio": float(error_ratio), "threshold": float(best_thr),
    "tp": int(len(tp)), "fn": int(len(fn)), "fp": int(len(fp)),
    "total_fraud": int(y.sum()),
    "tp_rate": float(len(tp)/y.sum()), "fn_rate": float(len(fn)/y.sum()),
}, open(f"{_export_dir}/metrics.json", "w"), indent=2)

# 2. Breakdown por tipo
_type_data = {}
for _t in ["TRANSFER", "CASH_OUT"]:
    _m = results["type"] == _t
    _tp_t = results[_m & (results["true"]==1) & (results["pred"]==1)]
    _fn_t = results[_m & (results["true"]==1) & (results["pred"]==0)]
    _total = len(_tp_t) + len(_fn_t)
    _type_data[_t] = {"tp": int(len(_tp_t)), "fn": int(len(_fn_t)),
                      "rate": float(len(_tp_t)/_total) if _total > 0 else 0}
_json.dump(_type_data, open(f"{_export_dir}/type_breakdown.json", "w"), indent=2)

# 3. Thresholds adaptativos
_adapt = {}
for _t, _thr in thresholds_by_type.items():
    _m = types_series == _t
    _yt = y[_m].values; _st = scores_ae[_m]; _pt = (_st > _thr).astype(int)
    _adapt[_t] = {"threshold": float(_thr),
                  "f1": float(f1_score(_yt, _pt, zero_division=0)),
                  "precision": float(precision_score(_yt, _pt, zero_division=0)),
                  "recall": float(recall_score(_yt, _pt, zero_division=0))}
_json.dump(_adapt, open(f"{_export_dir}/adaptive_thresholds.json", "w"), indent=2)

# 4. Valor em risco
_json.dump({
    "fn_total": float(_np.expm1(fn["log_amount"]).sum()),
    "fn_mean":  float(_np.expm1(fn["log_amount"]).mean()),
    "tp_total": float(_np.expm1(tp["log_amount"]).sum()),
    "tp_mean":  float(_np.expm1(tp["log_amount"]).mean()),
}, open(f"{_export_dir}/value_at_risk.json", "w"), indent=2)

# 5. Distribuição de scores (amostra)
_np.random.seed(42)
_idx_n = _np.where(y==0)[0]; _idx_f = _np.where(y==1)[0]
_samp_n = _np.random.choice(_idx_n, min(3000, len(_idx_n)), replace=False)
pd.DataFrame({
    "score_ae":  _np.concatenate([scores_ae[_samp_n],  scores_ae[_idx_f]]),
    "score_iso": _np.concatenate([scores_iso[_samp_n], scores_iso[_idx_f]]),
    "label": ["normal"]*len(_samp_n) + ["fraud"]*len(_idx_f),
}).to_csv(f"{_export_dir}/scores.csv", index=False)

# 6. Curvas ROC (downsampled)
_fpr_ae,  _tpr_ae,  _ = _roc_curve(y, scores_ae)
_fpr_iso, _tpr_iso, _ = _roc_curve(y, scores_iso)
_s = max(1, len(_fpr_ae)//300)
_json.dump({
    "ae":  {"fpr": _fpr_ae[::_s].tolist(),  "tpr": _tpr_ae[::_s].tolist(),  "auc": float(auc_ae)},
    "iso": {"fpr": _fpr_iso[::_s].tolist(), "tpr": _tpr_iso[::_s].tolist(), "auc": float(auc_iso)},
}, open(f"{_export_dir}/roc_curve.json", "w"), indent=2)

# 7. Curva PR (downsampled)
_s2 = max(1, len(precision)//300)
_json.dump({
    "precision": precision[::_s2].tolist(), "recall": recall[::_s2].tolist(),
    "auc_pr": float(pr_auc), "best_f1": float(f1_scores[best_idx]),
    "best_recall": float(recall[best_idx]), "best_precision": float(precision[best_idx]),
}, open(f"{_export_dir}/pr_curve.json", "w"), indent=2)

# 8. SHAP
if RUN_SHAP:
    _json.dump({c: float(_np.abs(iso_sv[:, i]).mean()) for i, c in enumerate(FEATURE_COLS)
                if i < iso_sv.shape[1]},
               open(f"{_export_dir}/shap_iso.json", "w"), indent=2)
    _json.dump({c: float(_np.abs(ae_sv[:, i]).mean()) for i, c in enumerate(FEATURE_COLS)
                if i < ae_sv.shape[1]},
               open(f"{_export_dir}/shap_ae.json", "w"), indent=2)

# 9. Features de erro
comparison.round(4).to_csv(f"{_export_dir}/error_features.csv")

print(f"✓ Dashboard data exportado para {_export_dir}")
print(f"  Arquivos: {sorted(_os.listdir(_export_dir))}")
if ON_KAGGLE:
    print("\n  Baixe a pasta dashboard_data e coloque em dashboard/data/ no repositório.")


# 10. Grafo interativo
if 'G' in dir() and 'df_metrics' in dir():
    # Resumo dos padrões detectados
    _n_fanout   = int((df_metrics["fan_out_score"] > 5).sum())
    _n_fanin    = int((df_metrics["fan_in_score"]  > 5).sum())
    _n_cycles   = int(df_metrics["cycle_flag"].sum())
    _n_hubs     = int((df_metrics["pagerank"] > df_metrics["pagerank"].quantile(0.99)).sum())
    _n_highrisk = int((df_metrics["graph_risk_score"] > 0.7).sum())
    _json.dump({
        "smurfing_fanout":     _n_fanout,
        "aggregation_fanin":   _n_fanin,
        "layering_cycles":     _n_cycles,
        "high_centrality_hubs": _n_hubs,
        "high_risk_score":     _n_highrisk,
        "n_nodes": int(G.number_of_nodes()),
        "n_edges": int(G.number_of_edges()),
    }, open(f"{_export_dir}/graph_summary.json", "w"), indent=2)

    # Top 30 contas por graph_risk_score
    _cols = ["account", "out_degree", "in_degree", "fan_out_score",
             "pagerank", "betweenness", "graph_risk_score", "is_fraud"]
    df_metrics.head(30)[_cols].round(4).to_csv(
        f"{_export_dir}/graph_top_hubs.csv", index=False)

    # HTML interativo PyVis
    from aml_detector.graph.analysis import export_pyvis
    export_pyvis(G, df_metrics, output_path=f"{_export_dir}/graph_interactive.html")
    print("✓ Grafo exportado")
else:
    print("⚠ Variáveis G / df_metrics não encontradas — célula de grafo não foi executada.")


In [ ]:
# Salva artefatos ao final (se treinou do zero)
if not LOAD_MODEL:
    save_artifacts(scaler, ae, best_thr, {
        "auc_autoencoder":    auc_ae,
        "auc_isolation_forest": auc_iso,
        "f1_fraud":           float(f1),
        "precision_fraud":    float(prec),
        "recall_fraud":       float(rec),
        "error_ratio":        float(error_ratio),
    })
    print("\n✓ Artefatos salvos em models/")
    print("  Para subir a API:")
    print("  uvicorn aml_detector.api.app:app --reload")